# Theoretical vs. Automatic Parameter and FLOPs Estimation

In [25]:
import os
import sys
from importlib import reload

current_dir = os.getcwd()
utilities_dir = os.path.join(current_dir, '../../utils')
os.chdir(current_dir)
if utilities_dir not in sys.path:
    sys.path.insert(0, utilities_dir)

import pinns_infinite
reload(pinns_infinite)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from torch.utils.flop_counter import FlopCounterMode

from pinns_infinite import build_models, build_models_KAN, set_seed

set_seed(42)
device = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')
input_shape = (1, 2)

In [26]:
def mlp_theoretical_params(input_size, output_size, hidden_layers, hidden_units):
    return (
        (input_size + 1) * hidden_units
        + hidden_layers * hidden_units * (hidden_units + 1)
        + (hidden_units + 1) * output_size
    )


def mlp_theoretical_flops(input_size, output_size, hidden_layers, hidden_units):
    return 2 * (
        input_size * hidden_units
        + hidden_layers * hidden_units**2
        + hidden_units * output_size
    )


def kan_theoretical_params(input_size, output_size, hidden_layers, hidden_units, grid_size, spline_order):
    weighted_sum_size = grid_size + spline_order + 2
    return weighted_sum_size * (
        (hidden_layers - 1) * hidden_units**2
        + (input_size + output_size) * hidden_units
    )


def kan_theoretical_flops(input_size, output_size, hidden_layers, hidden_units, grid_size, spline_order):
    # Only base_weight and spline_weight are used in a matmul; spline_scaler is an
    # elementwise multiply on the weight tensor and is invisible to matmul-based FLOP counters.
    matmul_weight_size = grid_size + spline_order + 1
    return 2 * matmul_weight_size * (
        (hidden_layers - 1) * hidden_units**2
        + (input_size + output_size) * hidden_units
    )

## Verify Against `FlopCounterMode` (One MLP and One KAN)

In [28]:
dummy_input = torch.randn(input_shape).to(device)


def measure_flops_with_torch(model):
    flop_counter = FlopCounterMode(display=False)
    with flop_counter:
        model(dummy_input)
    return flop_counter.get_total_flops()


INPUT_SIZE, OUTPUT_SIZE = 2, 1
HIDDEN_LAYERS, HIDDEN_UNITS = 3, 25
GRID_SIZE, SPLINE_ORDER = 5, 3

mlp_model, _ = build_models(device, hidden_layers=HIDDEN_LAYERS, hidden_units=HIDDEN_UNITS)
kan_model, _ = build_models_KAN(
    device,
    hidden_layers=HIDDEN_LAYERS,
    hidden_units=HIDDEN_UNITS,
    grid_size=GRID_SIZE,
    spline_order=SPLINE_ORDER,
)

results = pd.DataFrame([
    {
        "Model": "MLP",
        "Theoretical params": mlp_theoretical_params(INPUT_SIZE, OUTPUT_SIZE, HIDDEN_LAYERS, HIDDEN_UNITS),
        "Measured params": sum(p.numel() for p in mlp_model.parameters()),
        "Theoretical FLOPs": mlp_theoretical_flops(INPUT_SIZE, OUTPUT_SIZE, HIDDEN_LAYERS, HIDDEN_UNITS),
        "FlopCounterMode FLOPs": measure_flops_with_torch(mlp_model),
    },
    {
        "Model": "KAN",
        "Theoretical params": kan_theoretical_params(INPUT_SIZE, OUTPUT_SIZE, HIDDEN_LAYERS, HIDDEN_UNITS, GRID_SIZE, SPLINE_ORDER),
        "Measured params": sum(p.numel() for p in kan_model.parameters()),
        "Theoretical FLOPs": kan_theoretical_flops(INPUT_SIZE, OUTPUT_SIZE, HIDDEN_LAYERS, HIDDEN_UNITS, GRID_SIZE, SPLINE_ORDER),
        "FlopCounterMode FLOPs": measure_flops_with_torch(kan_model),
    },
])
results["Params rel. error (%)"] = (
    100 * (results["Theoretical params"] - results["Measured params"]).abs() / results["Measured params"]
).round(3)
results["FLOPs rel. error (%)"] = (
    100 * (results["Theoretical FLOPs"] - results["FlopCounterMode FLOPs"]).abs() / results["FlopCounterMode FLOPs"]
).round(3)
print(results.to_string(index=False))

Model  Theoretical params  Measured params  Theoretical FLOPs  FlopCounterMode FLOPs  Params rel. error (%)  FLOPs rel. error (%)
  MLP                2051             2051               3900                   3900                    0.0                   0.0
  KAN               13250            13250              23850                  23850                    0.0                   0.0
